# LeetCode 630 — Course Schedule III bằng Python

- Link: https://leetcode.com/problems/course-schedule-iii/
- Độ khó: **Hard**
- Chủ đề: Tham lam (Greedy), Heap (max-heap), Sắp xếp, Quy hoạch động (DP), Lập lịch theo hạn chót

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def scheduleCourse(self, courses: List[List[int]]) -> int:

```

## 1. Giới thiệu & mục tiêu học

Đây là bài **lập lịch có hạn chót (scheduling with deadlines)** kinh điển. Lời giải tối ưu ngắn đến bất ngờ (sắp xếp + heap, khoảng 10 dòng), nhưng để **tin** nó đúng, ta cần hiểu hai ý tưởng tham lam và chứng minh bằng lập luận trao đổi.

Sau notebook này bạn sẽ:

1. Biết kiểm tra **một tập môn có xếp lịch được không**: sắp theo hạn chót rồi học liên tục (quy tắc **EDF — Earliest Deadline First**).
2. Viết lời giải vét cạn tập con, rồi hai lời giải DP: **knapsack theo thời gian** và **"thời gian nhỏ nhất để học `j` môn"**.
3. Hiểu chiến lược tham lam **"học thử, nếu trễ thì bỏ môn dài nhất"** và cài đặt bằng **max-heap**.
4. Chứng minh tham lam bằng **lập luận trao đổi** và kiểm chứng bằng brute force có seed.
5. Biết cách truy vết ra **danh sách môn được chọn** và lịch học cụ thể.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `n` môn học trực tuyến, `courses[i] = [duration_i, lastDay_i]`: môn `i` cần học **liên tục** `duration_i` ngày và phải **hoàn thành trước hoặc vào ngày** `lastDay_i`. Bạn bắt đầu vào **ngày 1**, mỗi lúc chỉ học được **một môn**, và không được học hai môn cùng lúc. Trả về **số môn nhiều nhất** có thể hoàn thành.

Diễn đạt bằng thời gian: nếu học một dãy môn liên tiếp bắt đầu từ thời điểm 0, môn thứ `m` kết thúc ở thời điểm = tổng thời lượng các môn từ đầu tới nó; cần tổng này `<= lastDay` của môn đó.

**Ví dụ 1**

```text
Input:  courses = [[100,200],[200,1300],[1000,1250],[2000,3200]]
Output: 3
Giải thích: có 4 môn, học được nhiều nhất 3 môn:
  - Môn [100,200]:  học ngày 1–100,   xong ngày 100  (<= 200).
  - Môn [1000,1250]: học ngày 101–1100, xong ngày 1100 (<= 1250).
  - Môn [200,1300]: học ngày 1101–1300, xong ngày 1300 (<= 1300).
  - Môn [2000,3200]: nếu học sẽ xong ngày 3300 > 3200 — không kịp.
```

**Ví dụ 2**

```text
Input:  courses = [[1,2]]
Output: 1
```

**Ví dụ 3**

```text
Input:  courses = [[3,2],[4,3]]
Output: 0
Giải thích: môn nào cũng dài hơn hạn chót của nó.
```

**Ràng buộc**

- `1 <= courses.length <= 10^4`
- `1 <= duration_i, lastDay_i <= 10^4`

In [1]:
def check_order(order, courses):
    """Học các môn theo đúng thứ tự order (chỉ số), liên tục từ thời điểm 0. In và trả về có hợp lệ không."""
    t = 0
    for i in order:
        d, last = courses[i]
        t += d
        ok = t <= last
        print(f"  môn {courses[i]}: xong ở {t:>5} {'<=' if ok else '>'} {last} {'OK' if ok else 'TRỄ'}")
        if not ok:
            return False
    return True

ex1 = [[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]
print("Thứ tự 0, 2, 1:", check_order([0, 2, 1], ex1))
print("Thứ tự 0, 1, 2:", check_order([0, 1, 2], ex1))

  môn [100, 200]: xong ở   100 <= 200 OK
  môn [1000, 1250]: xong ở  1100 <= 1250 OK
  môn [200, 1300]: xong ở  1300 <= 1300 OK
Thứ tự 0, 2, 1: True
  môn [100, 200]: xong ở   100 <= 200 OK
  môn [200, 1300]: xong ở   300 <= 1300 OK
  môn [1000, 1250]: xong ở  1300 > 1250 TRỄ
Thứ tự 0, 1, 2: False


Cùng một tập môn `{0, 1, 2}` nhưng thứ tự `0, 1, 2` thì trễ, thứ tự `0, 2, 1` thì kịp — **thứ tự học quan trọng**. Mục 4 sẽ chỉ ra thứ tự tốt nhất cho một tập cố định.

## 3. Kiến thức nền

### 3.1. Sắp xếp theo khóa với `sort(key=...)`

`courses.sort(key=lambda c: c[1])` sắp theo hạn chót tăng dần. Sắp xếp của Python **ổn định** (stable): hai phần tử bằng khóa giữ thứ tự ban đầu.

In [2]:
cs = [[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]
print(sorted(cs, key=lambda c: c[1]))
print(sorted(cs, key=lambda c: (c[1], c[0])))    # khóa tuple: hạn chót, rồi thời lượng

[[100, 200], [1000, 1250], [200, 1300], [2000, 3200]]
[[100, 200], [1000, 1250], [200, 1300], [2000, 3200]]


### 3.2. Max-heap bằng `heapq` và số đối

`heapq` là **min-heap**. Muốn lấy phần tử **lớn nhất**, đẩy vào `-x` rồi lấy ra đổi dấu. `heappush`, `heappop` đều `O(log n)`; `h[0]` là đỉnh (nhỏ nhất).

In [3]:
import heapq
h = []
for d in [100, 1000, 200]:
    heapq.heappush(h, -d)
print("lớn nhất:", -h[0])
print("lấy ra:", -heapq.heappop(h), "| còn:", sorted(-x for x in h))

lớn nhất: 1000
lấy ra: 1000 | còn: [100, 200]


### 3.3. Duyệt mọi tập con bằng bitmask

Với `n` phần tử, mỗi số `mask` trong `0..2^n - 1` ứng với một tập con: phần tử `i` được chọn ⇔ bit `i` của `mask` bằng 1.

In [4]:
items = ["A", "B", "C"]
for mask in range(1 << 3):
    print(f"{mask:03b} ->", [items[i] for i in range(3) if mask >> i & 1])

000 -> []
001 -> ['A']
010 -> ['B']
011 -> ['A', 'B']
100 -> ['C']
101 -> ['A', 'C']
110 -> ['B', 'C']
111 -> ['A', 'B', 'C']


### 3.4. Mảng DP một chiều duyệt ngược (knapsack 0/1)

Trong knapsack 0/1, mỗi đồ vật dùng tối đa một lần → cập nhật `dp[t]` với `t` **giảm dần**, để `dp[t - w]` vẫn là giá trị **trước khi** xét đồ vật hiện tại.

In [5]:
weights, values, W = [2, 3], [3, 4], 5
dp = [0] * (W + 1)
for w, v in zip(weights, values):
    for t in range(W, w - 1, -1):
        dp[t] = max(dp[t], dp[t - w] + v)
print(dp)          # dp[5] = 7: lấy cả hai món

[0, 0, 3, 4, 4, 7]


### 3.5. `itertools.accumulate` — tổng dồn

`accumulate([3, 1, 4])` cho `3, 4, 8` — thời điểm kết thúc từng môn nếu học liên tục theo thứ tự đó.

In [6]:
from itertools import accumulate
durations = [100, 1000, 200]
print(list(accumulate(durations)))

[100, 1100, 1300]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Với một tập môn cố định: học theo hạn chót tăng dần (EDF)

**Mệnh đề:** một tập môn xếp lịch được ⇔ học chúng theo **hạn chót tăng dần**, liên tục từ thời điểm 0, thì môn nào cũng kịp.

**Vì sao?** Giả sử có một lịch hợp lệ mà hai môn liền nhau `X` rồi `Y` có `last_X > last_Y` ("lộn xộn"). Đổi chỗ thành `Y` rồi `X`:
- `Y` kết thúc **sớm hơn** trước (vì giờ học trước `X`) → vẫn kịp.
- `X` kết thúc đúng lúc `Y` kết thúc trước đây, mà lúc đó `<= last_Y < last_X` → vẫn kịp.
Đổi chỗ dần như sắp xếp nổi bọt, ta được lịch EDF vẫn hợp lệ. Ngoài ra, nghỉ giữa các môn không bao giờ có lợi.

### 4.2. Tính tay ví dụ 1

Sắp theo hạn chót: `[100,200], [1000,1250], [200,1300], [2000,3200]`. Tổng dồn: `100, 1100, 1300, 3300`. Ba môn đầu kịp, môn cuối trễ (3300 > 3200). Bỏ môn nào? Nếu bỏ **môn dài nhất** trong các môn đã chọn (2000 ngày, chính là môn cuối) thì thời gian quay về 1300 — nhỏ nhất có thể. Đáp án 3.

### 4.3. Tính tay ví dụ "cần thay môn"

`courses = [[5,5],[4,6],[2,6]]`. Sắp: `[5,5], [4,6], [2,6]`.
- Học `[5,5]`: xong ở 5 ✓.
- Thêm `[4,6]`: xong ở 9 > 6 ✗. Bỏ môn dài nhất (5) → còn `{[4,6]}`, thời gian 4.
- Thêm `[2,6]`: xong ở 6 ✓. Kết quả 2 môn: `[4,6]`, `[2,6]`.

Nếu "cứng đầu" giữ `[5,5]` thì chỉ được 1 môn. Bài học: **một môn dài chiếm chỗ của hai môn ngắn** — khi phải bỏ, bỏ môn dài nhất để dành thời gian.

In [7]:
def edf_feasible(subset):
    t = 0
    for d, last in sorted(subset, key=lambda c: c[1]):
        t += d
        if t > last:
            return False
    return True

print(edf_feasible([[100, 200], [200, 1300], [1000, 1250]]))
print(edf_feasible([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]))
print(edf_feasible([[4, 6], [2, 6]]), edf_feasible([[5, 5], [4, 6]]))

True
False
True False


## 5. Cách 1 — Brute Force: thử mọi tập con

**Ý tưởng.** Với mỗi tập con, kiểm tra xếp lịch được không bằng EDF (mục 4.1). Lấy tập lớn nhất xếp được.

In [8]:
def schedule_brute(courses):
    n = len(courses)
    best = 0
    for mask in range(1 << n):
        sub = [courses[i] for i in range(n) if mask >> i & 1]
        if len(sub) > best and edf_feasible(sub):
            best = len(sub)
    return best

print(schedule_brute(ex1), schedule_brute([[1, 2]]), schedule_brute([[3, 2], [4, 3]]), schedule_brute([[5, 5], [4, 6], [2, 6]]))
assert schedule_brute(ex1) == 3
assert schedule_brute([[1, 2]]) == 1
assert schedule_brute([[3, 2], [4, 3]]) == 0

3 1 0 2


**Độ phức tạp:** `2^n` tập con × `O(n log n)` kiểm tra → **`O(2^n · n log n)`**. `n = 20` đã ~ 1 triệu tập con; `n = 10^4` là không tưởng. Space `O(n)`.

**Nút thắt:** ta kiểm tra lại từ đầu cho từng tập con. Nhưng nhờ EDF, ta có thể xét các môn **theo thứ tự hạn chót** và quyết định "lấy / không lấy" từng môn — đó là DP; rồi thậm chí là tham lam.

In [9]:
import time, random
random.seed(0)
for n in (12, 15, 17):
    cs = [[random.randint(1, 10), random.randint(1, 40)] for _ in range(n)]
    t = time.perf_counter()
    schedule_brute(cs)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=12: 0.003s
n=15: 0.023s


n=17: 0.142s


## 6. Bước đột phá

**Insight 1 (EDF):** chỉ cần xét các môn theo **hạn chót tăng dần**; tập đã chọn luôn được học theo thứ tự đó. Khi xét môn mới (có hạn chót lớn nhất cho tới giờ), nó được xếp **cuối** lịch.

**Insight 2 (tham lam "thay môn dài nhất"):** Duyệt theo hạn chót. Luôn **thêm** môn mới vào tập. Nếu tổng thời gian vượt hạn chót của môn mới, **bỏ môn dài nhất** trong tập (có thể chính là môn mới).

Tại sao bỏ môn dài nhất? Sau khi bỏ, số môn quay về như trước khi thêm, nhưng **tổng thời gian nhỏ nhất có thể** → để lại nhiều chỗ nhất cho các môn sau.

**Invariant:** sau khi xử lý `m` môn đầu (theo hạn chót), tập `S` đang giữ
1. xếp lịch được,
2. có **số môn lớn nhất** có thể chọn từ `m` môn đầu,
3. trong các tập lớn nhất đó, có **tổng thời gian nhỏ nhất**.

**Quy tắc nếu…thì…:**
- Nếu thêm môn mới vẫn kịp thì giữ nguyên (số môn tăng 1).
- Nếu trễ thì bỏ môn có thời lượng lớn nhất (dùng max-heap để tìm nhanh).
- Sau khi bỏ, tổng thời gian chắc chắn `<=` tổng trước khi thêm, nên các môn còn lại vẫn kịp.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time |
|---|---|---|
| 1 | Brute force tập con (mục 5) | `O(2^n · n log n)` |
| 2 | DP knapsack theo thời gian | `O(n · D)`, `D` = hạn chót lớn nhất |
| 3 | DP "thời gian nhỏ nhất để học `j` môn" | `O(n^2)` |
| 4 | Tham lam + max-heap | `O(n log n)` |

### 7.1. Cách 2 — DP knapsack theo thời gian

**Giải thích.** Sắp theo hạn chót. `dp[t]` = số môn nhiều nhất chọn được (trong các môn đã xét) sao cho **tổng thời gian đúng bằng `t`** và lịch EDF hợp lệ (`-inf` nếu không đạt được). Môn `(d, last)` được thêm vào cuối lịch có tổng `t` ⇔ `t <= last`:

```text
với t từ last xuống d:   dp[t] = max(dp[t], dp[t - d] + 1)
```

Duyệt `t` giảm dần (knapsack 0/1). Đáp án `max(dp)`.

In [10]:
def schedule_knapsack(courses):
    cs = sorted(courses, key=lambda c: c[1])
    D = max(last for _, last in cs)
    NEG = float('-inf')
    dp = [0] + [NEG] * D               # dp[t]: tổng thời gian đúng bằng t
    for d, last in cs:
        for t in range(last, d - 1, -1):
            if dp[t - d] + 1 > dp[t]:
                dp[t] = dp[t - d] + 1
    return max(dp)

print(schedule_knapsack(ex1), schedule_knapsack([[5, 5], [4, 6], [2, 6]]), schedule_knapsack([[3, 2], [4, 3]]))
assert schedule_knapsack(ex1) == 3

3 2 0


**Đọc code:**

- `dp[0] = 0`: chưa học gì, tổng 0. Các `t` khác ban đầu không đạt được.
- Vòng `t` chạy từ `last` (môn mới phải xong trước `last`) xuống `d`.
- Vì xét theo hạn chót tăng dần, môn mới luôn đứng cuối lịch EDF — điều kiện `t <= last` là đủ; các môn trước đã kịp với hạn chót của chúng từ trước.

**Code nộp LeetCode (Cách 2):** `n · D` có thể tới `10^8` → **TLE** trong Python. Dùng để hiểu mô hình DP.

In [11]:
from typing import List

class SolutionKnapsack:
    def scheduleCourse(self, courses: List[List[int]]) -> int:
        courses.sort(key=lambda c: c[1])
        D = courses[-1][1]
        dp = [0] + [-1] * D                # -1: không đạt được
        for d, last in courses:
            for t in range(last, d - 1, -1):
                if dp[t - d] >= 0 and dp[t - d] + 1 > dp[t]:
                    dp[t] = dp[t - d] + 1
        return max(dp)

assert SolutionKnapsack().scheduleCourse([row[:] for row in ex1]) == 3
print("SolutionKnapsack OK (chỉ cho dữ liệu nhỏ)")

SolutionKnapsack OK (chỉ cho dữ liệu nhỏ)


**Độ phức tạp Cách 2:** Time `O(n log n + n · D)`, Space `O(D)`.

### 7.2. Cách 3 — DP "thời gian nhỏ nhất để học `j` môn"

**Giải thích.** Đổi vai trò: thay vì cố định thời gian, cố định **số môn**. `best[j]` = **tổng thời gian nhỏ nhất** để học đúng `j` môn (trong các môn đã xét, theo EDF). Thêm môn `(d, last)`:

```text
với j từ (số môn đã xét) xuống 1:
    nếu best[j-1] + d <= last:  best[j] = min(best[j], best[j-1] + d)
```

Đáp án = `j` lớn nhất có `best[j]` hữu hạn. Vì số môn `<= n`, độ phức tạp `O(n^2)` không phụ thuộc giá trị hạn chót.

In [12]:
def schedule_min_time(courses):
    cs = sorted(courses, key=lambda c: c[1])
    INF = float('inf')
    best = [0] + [INF] * len(cs)        # best[j]: tổng thời gian nhỏ nhất cho j môn
    for idx, (d, last) in enumerate(cs, 1):
        for j in range(idx, 0, -1):
            cand = best[j - 1] + d
            if cand <= last and cand < best[j]:
                best[j] = cand
    return max(j for j in range(len(best)) if best[j] < INF)

print(schedule_min_time(ex1), schedule_min_time([[5, 5], [4, 6], [2, 6]]), schedule_min_time([[3, 2], [4, 3]]))
assert schedule_min_time(ex1) == 3

3 2 0


**Đọc code:**

- `best[0] = 0`: học 0 môn mất 0 ngày.
- `j` duyệt giảm dần để mỗi môn chỉ dùng một lần (giống knapsack).
- `cand <= last`: môn mới đứng cuối lịch, phải kịp hạn chót của nó.
- Nhận xét: `best[j]` tăng theo `j` — trạng thái này chính là thông tin mà tham lam (Cách 4) "nén" lại thành một tập duy nhất.

**Code nộp LeetCode (Cách 3):** `n^2 / 2 = 5 · 10^7` phép với `n = 10^4` → vẫn chậm trong Python, có thể TLE.

In [13]:
class SolutionMinTime:
    def scheduleCourse(self, courses: List[List[int]]) -> int:
        courses.sort(key=lambda c: c[1])
        INF = float('inf')
        best = [0] + [INF] * len(courses)
        top = 0                                   # số môn lớn nhất đạt được tới giờ
        for d, last in courses:
            for j in range(top + 1, 0, -1):
                cand = best[j - 1] + d
                if cand <= last and cand < best[j]:
                    best[j] = cand
            if best[top + 1] < INF:
                top += 1
        return top

assert SolutionMinTime().scheduleCourse([row[:] for row in ex1]) == 3
assert SolutionMinTime().scheduleCourse([[5, 5], [4, 6], [2, 6]]) == 2
print("SolutionMinTime OK")

SolutionMinTime OK


Bản nộp chỉ duyệt `j` tới `top + 1` (không thể học nhiều hơn `top + 1` môn sau khi thêm một môn) — nhanh hơn đáng kể khi đáp án nhỏ.

**Độ phức tạp Cách 3:** Time `O(n log n + n · ans)` ≤ `O(n^2)`, Space `O(n)`.

### 7.3. Cách 4 — Tham lam + max-heap

**Giải thích.** Cài đặt Insight 2:

```text
sắp courses theo lastDay
time = 0; heap = []                 (max-heap thời lượng các môn đang chọn)
với mỗi (d, last):
    time += d; push d
    nếu time > last: time -= pop_max()
trả về số phần tử trong heap
```

Chỉ cần **một** lần bỏ là đủ: trước khi thêm, tập kịp hạn (`time_cũ <= last_trước <= last`); bỏ môn dài nhất (`>= d`) làm `time <= time_cũ <= last`.

In [14]:
import heapq

def schedule_greedy(courses):
    cs = sorted(courses, key=lambda c: c[1])
    time_used = 0
    heap = []                                     # lưu -duration
    for d, last in cs:
        time_used += d
        heapq.heappush(heap, -d)
        if time_used > last:
            time_used += heapq.heappop(heap)      # cộng số âm = trừ môn dài nhất
    return len(heap)

print(schedule_greedy(ex1), schedule_greedy([[5, 5], [4, 6], [2, 6]]), schedule_greedy([[1, 2]]), schedule_greedy([[3, 2], [4, 3]]))
assert schedule_greedy(ex1) == 3

3 2 1 0


**Đọc code:**

- `heapq.heappush(heap, -d)`: lưu số âm để đỉnh heap là môn **dài nhất**.
- `time_used += heapq.heappop(heap)`: `heappop` trả `-d_max`, cộng vào là trừ `d_max`.
- `if` (không phải `while`): một lần bỏ đã đủ như giải thích ở trên.
- Môn có `d > last` (không bao giờ kịp) tự động bị bỏ: thêm vào thì trễ, và nó là môn dài nhất hoặc một môn dài hơn bị bỏ thay — tập vẫn tốt.

**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionGreedy:
    def scheduleCourse(self, courses: List[List[int]]) -> int:
        courses.sort(key=lambda c: c[1])
        heap, total = [], 0
        for d, last in courses:
            total += d
            heapq.heappush(heap, -d)
            if total > last:
                total += heapq.heappop(heap)
        return len(heap)

assert SolutionGreedy().scheduleCourse([row[:] for row in ex1]) == 3
assert SolutionGreedy().scheduleCourse([[1, 2]]) == 1
assert SolutionGreedy().scheduleCourse([[3, 2], [4, 3]]) == 0
print("SolutionGreedy OK")

SolutionGreedy OK


**Độ phức tạp Cách 4:** sắp xếp `O(n log n)` + mỗi môn tối đa một lần push và một lần pop `O(log n)` → **`O(n log n)`**, Space `O(n)`.

## 8. TRACE từng bước

Trace **Cách 4** với `courses = [[5,5],[4,6],[2,6],[3,8],[1,3]]`. Sau khi sắp theo hạn chót: `[1,3], [5,5], [4,6], [2,6], [3,8]`.

| bước | môn (d, last) | time sau khi thêm | trễ? | bỏ | heap (thời lượng) | time |
|---|---|---|---|---|---|---|
| 1 | (1, 3) | 1 | không | — | {1} | 1 |
| 2 | (5, 5) | 6 | 6 > 5 | 5 | {1} | 1 |
| 3 | (4, 6) | 5 | không | — | {1, 4} | 5 |
| 4 | (2, 6) | 7 | 7 > 6 | 4 | {1, 2} | 3 |
| 5 | (3, 8) | 6 | không | — | {1, 2, 3} | 6 |

Đáp án 3: các môn `[1,3], [2,6], [3,8]`, lịch xong ở 1, 3, 6.

In [16]:
def trace_greedy(courses):
    cs = sorted(courses, key=lambda c: c[1])
    print("sau khi sắp theo lastDay:", cs)
    total, heap = 0, []
    for step, (d, last) in enumerate(cs, 1):
        total += d
        heapq.heappush(heap, -d)
        msg = f"bước {step}: thêm ({d}, {last}) -> time = {total}"
        if total > last:
            removed = -heapq.heappop(heap)
            total -= removed
            msg += f" > {last}: TRỄ, bỏ môn dài nhất ({removed}) -> time = {total}"
        else:
            msg += f" <= {last}: kịp"
        print(msg, "| heap =", sorted(-x for x in heap))
    print("đáp án:", len(heap))
    return len(heap)

assert trace_greedy([[5, 5], [4, 6], [2, 6], [3, 8], [1, 3]]) == 3

sau khi sắp theo lastDay: [[1, 3], [5, 5], [4, 6], [2, 6], [3, 8]]
bước 1: thêm (1, 3) -> time = 1 <= 3: kịp | heap = [1]
bước 2: thêm (5, 5) -> time = 6 > 5: TRỄ, bỏ môn dài nhất (5) -> time = 1 | heap = [1]
bước 3: thêm (4, 6) -> time = 5 <= 6: kịp | heap = [1, 4]
bước 4: thêm (2, 6) -> time = 7 > 6: TRỄ, bỏ môn dài nhất (4) -> time = 3 | heap = [1, 2]
bước 5: thêm (3, 8) -> time = 6 <= 8: kịp | heap = [1, 2, 3]
đáp án: 3


Trace **Cách 3** cùng dữ liệu: in mảng `best[j]` (thời gian nhỏ nhất cho `j` môn) sau mỗi môn. Ta thấy `best` chứa nhiều thông tin hơn tham lam cần, nhưng đáp án giống nhau.

In [17]:
def trace_min_time(courses):
    cs = sorted(courses, key=lambda c: c[1])
    INF = float('inf')
    best = [0] + [INF] * len(cs)
    fmt = lambda arr: "[" + ", ".join("∞" if v == INF else str(v) for v in arr) + "]"
    for idx, (d, last) in enumerate(cs, 1):
        for j in range(idx, 0, -1):
            cand = best[j - 1] + d
            if cand <= last and cand < best[j]:
                best[j] = cand
        print(f"sau môn ({d}, {last}): best = {fmt(best)}")
    ans = max(j for j in range(len(best)) if best[j] < INF)
    print("đáp án:", ans)
    return ans

assert trace_min_time([[5, 5], [4, 6], [2, 6], [3, 8], [1, 3]]) == 3

sau môn (1, 3): best = [0, 1, ∞, ∞, ∞, ∞]
sau môn (5, 5): best = [0, 1, ∞, ∞, ∞, ∞]
sau môn (4, 6): best = [0, 1, 5, ∞, ∞, ∞]
sau môn (2, 6): best = [0, 1, 3, ∞, ∞, ∞]
sau môn (3, 8): best = [0, 1, 3, 6, ∞, ∞]
đáp án: 3


## 9. Code nộp LeetCode

Chọn Cách 4 (tham lam + max-heap): `O(n log n)`, ngắn gọn.

In [18]:
import heapq
from typing import List

class Solution:
    def scheduleCourse(self, courses: List[List[int]]) -> int:
        courses.sort(key=lambda c: c[1])          # EDF: xét theo hạn chót tăng dần
        heap = []                                 # max-heap thời lượng (lưu số âm)
        total = 0                                 # tổng thời gian các môn đang chọn
        for duration, last_day in courses:
            total += duration
            heapq.heappush(heap, -duration)
            if total > last_day:                  # trễ hạn -> bỏ môn dài nhất
                total += heapq.heappop(heap)
        return len(heap)

sol = Solution()
assert sol.scheduleCourse([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]) == 3
assert sol.scheduleCourse([[1, 2]]) == 1
assert sol.scheduleCourse([[3, 2], [4, 3]]) == 0
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

Xét các môn theo hạn chót tăng dần `c_1, c_2, ..., c_n`. Gọi `S_m` là tập tham lam giữ sau khi xử lý `m` môn đầu, `T(S)` là tổng thời lượng của `S`.

**Khẳng định (invariant):** `S_m` xếp lịch được, `|S_m| = k_m` là số môn lớn nhất chọn được từ `{c_1..c_m}`, và `T(S_m)` nhỏ nhất trong mọi tập xếp được có `k_m` môn. (Thực ra mạnh hơn: với mọi `j <= k_m`, `j` môn ngắn nhất của `S_m` có tổng `<=` mọi tập `j` môn xếp được — nhưng ta chỉ cần ý chính.)

**Bước quy nạp — thêm `c = (d, last)`:**

- *Trường hợp kịp* (`T(S_{m-1}) + d <= last`): `S_m = S_{m-1} ∪ {c}` xếp được (EDF, `c` đứng cuối). Không thể chọn `k_{m-1} + 2` môn từ `m` môn đầu (nếu được, bỏ `c` sẽ còn `k_{m-1}+1` môn từ `m-1` môn đầu — mâu thuẫn). Tổng thời gian nhỏ nhất của tập `k_{m-1}+1` môn: tập đó phải chứa `c` (nếu không đã mâu thuẫn), bỏ `c` còn tập `k_{m-1}` môn có tổng `>= T(S_{m-1})` → tổng `>= T(S_{m-1}) + d`. ✓
- *Trường hợp trễ*: không có tập `k_{m-1} + 1` môn xếp được (tập như vậy phải chứa `c`, phần còn lại có tổng `>= T(S_{m-1})`, nên `c` xong ở thời điểm `>= T(S_{m-1}) + d > last`). Vậy `k_m = k_{m-1}`. Tham lam bỏ môn dài nhất `x` của `S_{m-1} ∪ {c}`: tổng mới `T(S_{m-1}) + d - d_x <= T(S_{m-1})` (vì `d_x >= d`) → các hạn chót đều thỏa (mọi tiền tố của lịch EDF chỉ ngắn đi hoặc giữ nguyên). Tổng này là nhỏ nhất vì: trong tập `S_{m-1} ∪ {c}` có `k+1` môn, bỏ đi môn dài nhất cho tổng nhỏ nhất; và mọi tập `k` môn xếp được khác từ `m` môn đầu có tổng không nhỏ hơn (lập luận trao đổi với `S_{m-1}`). ✓

Kết thúc: `|S_n|` là đáp án. ∎

**DP (Cách 2, 3)** đúng vì nhờ EDF, mỗi tập xếp được tương ứng một dãy quyết định "lấy/không lấy" theo thứ tự hạn chót, với điều kiện môn được lấy kết thúc trước hạn chót của nó — chính là công thức chuyển trạng thái.

## 11. Phân tích Time/Space

| Cách | Time | Space | Với `n = D = 10^4` |
|---|---|---|---|
| 1. Brute force | `O(2^n · n log n)` | `O(n)` | không thể |
| 2. Knapsack thời gian | `O(n · D)` | `O(D)` | `10^8` — TLE trong Python |
| 3. Thời gian nhỏ nhất cho `j` môn | `O(n · ans)` ≤ `O(n^2)` | `O(n)` | tới `5·10^7` — chậm |
| 4. Tham lam + heap | `O(n log n)` | `O(n)` | rất nhanh |

In [19]:
random.seed(1)
big = [[random.randint(1, 100), random.randint(1, 10**4)] for _ in range(10**4)]
for name, cls in [("Greedy", SolutionGreedy), ("Solution", Solution)]:
    t = time.perf_counter()
    r = cls().scheduleCourse([c[:] for c in big])
    print(f"{name:>9}: {r}  {time.perf_counter() - t:.3f}s")
mid = big[:500]
for name, cls in [("MinTime", SolutionMinTime), ("Knapsack", SolutionKnapsack)]:
    t = time.perf_counter()
    r = cls().scheduleCourse([c[:] for c in mid])
    print(f"{name:>9}: {r}  {time.perf_counter() - t:.3f}s  (chỉ n = 500)")
assert SolutionGreedy().scheduleCourse([c[:] for c in mid]) == SolutionMinTime().scheduleCourse([c[:] for c in mid])

   Greedy: 1341  0.006s
 Solution: 1341  0.006s
  MinTime: 306  0.004s  (chỉ n = 500)


 Knapsack: 306  0.143s  (chỉ n = 500)


## 12. Bộ test

In [20]:
impls = {
    "knapsack": schedule_knapsack,
    "min_time": schedule_min_time,
    "greedy": schedule_greedy,
    "SolutionKnapsack": lambda c: SolutionKnapsack().scheduleCourse(c),
    "SolutionMinTime": lambda c: SolutionMinTime().scheduleCourse(c),
    "SolutionGreedy": lambda c: SolutionGreedy().scheduleCourse(c),
    "Solution": lambda c: Solution().scheduleCourse(c),
}

def check(courses, exp=None):
    exp = schedule_brute(courses) if exp is None else exp
    for name, f in impls.items():
        got = f([c[:] for c in courses])          # truyền bản sao vì một số cách sort tại chỗ
        assert got == exp, (name, courses, got, exp)

# Ví dụ đề
check([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]], 3)
check([[1, 2]], 1)
check([[3, 2], [4, 3]], 0)

# Edge case
check([[1, 1]], 1)                                # vừa khít
check([[2, 1]], 0)                                # quá dài
check([[1, 5]] * 5, 5)                            # nhiều môn giống nhau, vừa đủ
check([[1, 4]] * 6, 4)                            # nhiều môn giống nhau, thừa
check([[5, 5], [4, 6], [2, 6]], 2)                # cần thay môn dài
check([[7, 17], [3, 12], [10, 20], [9, 10], [5, 20], [10, 19], [4, 18]], 4)
check([[10000, 10000]], 1)
print("Ví dụ đề + edge case OK")

# Ngẫu nhiên có seed, đối chiếu brute force
random.seed(630)
for _ in range(400):
    n = random.randint(1, 10)
    maxd = random.randint(1, 10)
    cs = [[random.randint(1, maxd), random.randint(1, 30)] for _ in range(n)]
    check(cs)
print("400 test ngẫu nhiên khớp brute force")

# Lớn hơn: các cách đa thức khớp nhau
for _ in range(30):
    n = random.randint(20, 200)
    cs = [[random.randint(1, 50), random.randint(1, 600)] for _ in range(n)]
    vals = {f([c[:] for c in cs]) for f in impls.values()}
    assert len(vals) == 1, cs
print("30 test lớn: các cách khớp nhau")

Ví dụ đề + edge case OK


400 test ngẫu nhiên khớp brute force


30 test lớn: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Sắp theo thời lượng (học môn ngắn trước)

In [21]:
def wrong_sort_by_duration(courses):
    total, cnt = 0, 0
    for d, last in sorted(courses):               # SAI: sắp theo duration
        if total + d <= last:
            total += d
            cnt += 1
    return cnt

cs = [[2, 2], [1, 3], [2, 4]]      # hoặc tự tìm phản ví dụ bằng máy như dưới
random.seed(1)
found = None
for _ in range(5000):
    t = [[random.randint(1, 5), random.randint(1, 10)] for _ in range(random.randint(1, 5))]
    if wrong_sort_by_duration(t) != Solution().scheduleCourse([c[:] for c in t]):
        if found is None or len(t) < len(found):
            found = t
assert found is not None
print("phản ví dụ:", found, "| sai:", wrong_sort_by_duration(found), "| đúng:", Solution().scheduleCourse([c[:] for c in found]))

phản ví dụ: [[3, 4], [2, 8]] | sai: 1 | đúng: 2


**Vì sao sai:** học môn ngắn trước có thể làm môn có hạn chót sớm bị trễ. Thứ tự học đúng cho một tập cố định là **theo hạn chót** (EDF).

**Code đúng:** sắp theo `lastDay`.

### Lỗi 2 — Sắp theo hạn chót nhưng chỉ "lấy nếu kịp", không bao giờ thay môn

In [22]:
def wrong_no_swap(courses):
    total, cnt = 0, 0
    for d, last in sorted(courses, key=lambda c: c[1]):
        if total + d <= last:
            total += d
            cnt += 1
    return cnt

print("sai:", wrong_no_swap([[5, 5], [4, 6], [2, 6]]), "| đúng:", Solution().scheduleCourse([[5, 5], [4, 6], [2, 6]]))

sai: 1 | đúng: 2


**Vì sao sai:** môn `[5,5]` được lấy trước (kịp), nhưng nó chiếm 5 ngày, chặn cả `[4,6]` lẫn `[2,6]`. Phải cho phép "đổi" môn dài lấy môn ngắn hơn.

**Code đúng:** luôn thêm, nếu trễ thì bỏ môn dài nhất (heap).

### Lỗi 3 — Dùng min-heap (bỏ môn **ngắn** nhất)

In [23]:
def wrong_min_heap(courses):
    total, heap = 0, []
    for d, last in sorted(courses, key=lambda c: c[1]):
        total += d
        heapq.heappush(heap, d)                   # SAI: min-heap
        if total > last:
            total -= heapq.heappop(heap)          # bỏ môn ngắn nhất
    return len(heap)

print("sai:", wrong_min_heap([[5, 5], [4, 6], [2, 6]]), "| đúng:", Solution().scheduleCourse([[5, 5], [4, 6], [2, 6]]))

sai: 1 | đúng: 2


**Vì sao sai:** bỏ môn ngắn nhất giảm thời gian ít nhất → có thể **vẫn trễ** (tập không còn hợp lệ!) và để lại ít chỗ cho môn sau. `heapq` mặc định là min-heap — phải lưu số âm.

**Code đúng:** `heappush(heap, -d)` và `total += heappop(heap)`.

### Lỗi 4 — Kiểm tra trễ **trước** khi thêm rồi bỏ qua môn mới

In [24]:
def wrong_check_before(courses):
    total, heap = 0, []
    for d, last in sorted(courses, key=lambda c: c[1]):
        if total + d <= last:
            total += d
            heapq.heappush(heap, -d)
        elif heap and -heap[0] > d and total - (-heap[0]) + d <= last:
            total += heapq.heappop(heap) + d      # thay môn dài nhất bằng môn mới
            heapq.heappush(heap, -d)
    return len(heap)

# Cách này thật ra ĐÚNG (tương đương), nhưng dễ viết sai điều kiện. Kiểm tra:
random.seed(7)
for _ in range(2000):
    t = [[random.randint(1, 6), random.randint(1, 12)] for _ in range(random.randint(1, 7))]
    assert wrong_check_before(t) == Solution().scheduleCourse([c[:] for c in t])
print("Biến thể 'thay nếu có lợi' cho cùng kết quả — nhưng code dài và nhiều điều kiện hơn.")

Biến thể 'thay nếu có lợi' cho cùng kết quả — nhưng code dài và nhiều điều kiện hơn.


**Bài học:** cách "chỉ thay khi môn mới ngắn hơn môn dài nhất" là tương đương, nhưng có 3 điều kiện dễ nhầm (`heap` rỗng, so sánh `>`, kiểm tra kịp sau khi thay). Cách "luôn thêm, trễ thì bỏ max" gọn và khó sai hơn.

### Lỗi 5 — Quên rằng `sort` sửa danh sách gốc

`courses.sort(...)` sắp **tại chỗ**. Trong bài LeetCode không sao, nhưng khi tự test nhiều cách trên cùng một list, cách sau nhận dữ liệu đã bị sắp lại. Ở mục 12 ta luôn truyền `[c[:] for c in courses]`.

## 14. Mẹo & nhận dạng pattern

- **"Chọn nhiều việc nhất, mỗi việc có thời lượng và hạn chót, làm tuần tự"** → sắp theo hạn chót + max-heap "bỏ việc dài nhất".
- **Một tập cố định thì làm theo hạn chót sớm nhất (EDF)** — kết quả dùng lại rất nhiều trong bài lập lịch.
- **Tham lam có "hối hận"** (regret greedy): luôn nhận, sau đó nếu vi phạm thì bỏ phần tử tệ nhất — heap giúp tìm phần tử tệ nhất trong `O(log n)`. Cùng họ: IPO (502), Maximum Performance of a Team (1383), Furthest Building You Can Reach (1642).
- Khi DP có chiều "giá trị lớn" (thời gian tới `10^4`), thử **đổi vai trò**: DP theo số phần tử, lưu giá trị nhỏ nhất (Cách 3).
- Luôn kiểm chứng tham lam bằng brute force tập con với `n <= 10`.

## 15. Biến thể

1. **Có trọng số (mỗi môn có điểm, tối đa tổng điểm):** tham lam không còn đúng → DP knapsack theo thời gian (Cách 2 thay `+1` bằng `+w`).
2. **Course Schedule I/II (LeetCode 207/210):** ràng buộc thứ tự tiên quyết → sắp xếp tô-pô (khác hẳn).
3. **Có thời điểm bắt đầu sớm nhất (release time):** bài toán khó hơn nhiều (NP-khó trong trường hợp tổng quát).
4. **Furthest Building You Can Reach (LeetCode 1642):** cùng kiểu "dùng tạm, vượt ngân sách thì đổi phần tốn nhất".
5. **Trả về danh sách môn được chọn** (Bài tập 5).

In [25]:
# Biến thể 1: mỗi môn có điểm w, tối đa tổng điểm (DP knapsack theo thời gian)
def max_weight_schedule(courses_w):
    cs = sorted(courses_w, key=lambda c: c[1])
    D = max(c[1] for c in cs)
    dp = [0] + [-1] * D
    for d, last, w in cs:
        for t in range(last, d - 1, -1):
            if dp[t - d] >= 0 and dp[t - d] + w > dp[t]:
                dp[t] = dp[t - d] + w
    return max(dp)

def max_weight_brute(courses_w):
    n, best = len(courses_w), 0
    for mask in range(1 << n):
        sub = [courses_w[i] for i in range(n) if mask >> i & 1]
        if edf_feasible([[d, l] for d, l, _ in sub]):
            best = max(best, sum(w for _, _, w in sub))
    return best

random.seed(15)
for _ in range(200):
    cw = [[random.randint(1, 5), random.randint(1, 15), random.randint(1, 9)] for _ in range(random.randint(1, 8))]
    assert max_weight_schedule(cw) == max_weight_brute(cw)
print("Biến thể có trọng số: DP khớp brute force; ví dụ:", max_weight_schedule([[5, 5, 10], [4, 6, 3], [2, 6, 3]]))

Biến thể có trọng số: DP khớp brute force; ví dụ: 10


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi tập con + kiểm tra EDF | `O(2^n · n log n)` | `O(n)` | kiểm chứng `n <= 15` |
| 2. Knapsack thời gian | `dp[t]` = số môn max với tổng `t` | `O(n · D)` | `O(D)` | biến thể có trọng số |
| 3. Thời gian nhỏ nhất cho `j` môn | `best[j]` min tổng | `O(n · ans)` | `O(n)` | khi `D` lớn mà `n` vừa phải |
| 4. Tham lam + heap | thêm, trễ thì bỏ môn dài nhất | `O(n log n)` | `O(n)` | **nộp bài** |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `scheduleCourse([[2,2],[2,3],[2,3]])` và `scheduleCourse([[1,2],[2,3],[3,4]])`?

**Bài 2 (tính tay).** Trace Cách 4 (bảng như mục 8) cho `[[7,17],[3,12],[10,20],[9,10],[5,20],[10,19],[4,18]]`.

**Bài 3 (điền TODO).** Hoàn thiện `feasible_in_order(courses)` trả về `True` nếu học **đúng theo thứ tự đã cho** thì mọi môn kịp hạn (không sắp lại).

**Bài 4 (tự viết lại).** Tự viết lại Cách 4 (`my_schedule`) mà không nhìn code.

**Bài 5 (mini challenge).** Viết `chosen_courses(courses)` trả về **danh sách chỉ số** (theo input gốc) của các môn được chọn bởi tham lam, kèm lịch `(chỉ số, ngày bắt đầu, ngày kết thúc)` theo EDF. Kiểm tra lịch hợp lệ.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def feasible_in_order(courses):
    # TODO: cộng dồn thời lượng, so với lastDay từng môn
    pass

# Bài 4 — TODO
def my_schedule(courses):
    # TODO: sort theo lastDay; heap số âm; thêm, trễ thì pop
    pass

# Bài 5 — TODO
def chosen_courses(courses):
    # TODO: heap lưu (-duration, index); cuối cùng lấy các index còn lại, sắp theo lastDay để dựng lịch
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: sắp theo hạn chót rồi cộng dồn.
- Mức 2: `[[2,2],[2,3],[2,3]]`: sau môn đầu `time = 2`; thêm môn thứ hai → 4 > 3, bỏ một môn dài 2; môn thứ ba cũng vậy.
- Mức 3: `[[1,2],[2,3],[3,4]]`: `1` kịp; `1+2 = 3` kịp; `3+3 = 6 > 4` → bỏ môn dài nhất (3).

**Bài 2.**
- Mức 1: sắp theo hạn chót: `[9,10], [3,12], [7,17], [4,18], [10,19], [10,20], [5,20]`.
- Mức 2: theo dõi `time` và heap sau mỗi bước.
- Mức 3: những lần trễ xảy ra ở môn `[7,17]`, `[10,19]`, `[10,20]`.

**Bài 3.**
- Mức 1: `t = 0`; với mỗi môn `t += d`.
- Mức 2: nếu `t > last` trả `False`.
- Mức 3: hết vòng trả `True`.

**Bài 4.**
- Mức 1: `courses.sort(key=lambda c: c[1])`.
- Mức 2: `heappush(h, -d)`, `total += d`.
- Mức 3: `if total > last: total += heappop(h)`.

**Bài 5.**
- Mức 1: đẩy `(-d, i)` vào heap.
- Mức 2: các chỉ số còn trong heap là tập được chọn.
- Mức 3: sắp chúng theo `lastDay`, cộng dồn để ra ngày bắt đầu/kết thúc (ngày bắt đầu = thời điểm trước + 1).

## 19. Đáp án tham khảo

**Bài 1:** `1` và `2`.

**Bài 2:** đáp án 4. Các bước: `[9,10]` (t=9) → `[3,12]` (t=12) → `[7,17]` t=19 > 17, bỏ 9 → t=10 → `[4,18]` t=14 → `[10,19]` t=24 > 19, bỏ 10 → t=14 → `[10,20]` t=24 > 20, bỏ 10 → t=14 → `[5,20]` t=19 ✓. Còn `{3, 7, 4, 5}`.

In [27]:
assert Solution().scheduleCourse([[2, 2], [2, 3], [2, 3]]) == 1
assert Solution().scheduleCourse([[2, 2], [2, 3], [2, 4]]) == 2   # thêm: hạn 4 thì kịp 2 môn
assert Solution().scheduleCourse([[1, 2], [2, 3], [3, 4]]) == 2
assert trace_greedy([[7, 17], [3, 12], [10, 20], [9, 10], [5, 20], [10, 19], [4, 18]]) == 4

# Bài 3
def feasible_in_order(courses):
    t = 0
    for d, last in courses:
        t += d
        if t > last:
            return False
    return True

assert feasible_in_order([[100, 200], [1000, 1250], [200, 1300]])
assert not feasible_in_order([[100, 200], [200, 1300], [1000, 1250]])

# Bài 4
def my_schedule(courses):
    h, total = [], 0
    for d, last in sorted(courses, key=lambda c: c[1]):
        heapq.heappush(h, -d)
        total += d
        if total > last:
            total += heapq.heappop(h)
    return len(h)

random.seed(4)
for _ in range(300):
    cs = [[random.randint(1, 8), random.randint(1, 25)] for _ in range(random.randint(1, 9))]
    assert my_schedule(cs) == schedule_brute(cs)
print("Bài 1–4 OK")

sau khi sắp theo lastDay: [[9, 10], [3, 12], [7, 17], [4, 18], [10, 19], [10, 20], [5, 20]]
bước 1: thêm (9, 10) -> time = 9 <= 10: kịp | heap = [9]
bước 2: thêm (3, 12) -> time = 12 <= 12: kịp | heap = [3, 9]
bước 3: thêm (7, 17) -> time = 19 > 17: TRỄ, bỏ môn dài nhất (9) -> time = 10 | heap = [3, 7]
bước 4: thêm (4, 18) -> time = 14 <= 18: kịp | heap = [3, 4, 7]
bước 5: thêm (10, 19) -> time = 24 > 19: TRỄ, bỏ môn dài nhất (10) -> time = 14 | heap = [3, 4, 7]
bước 6: thêm (10, 20) -> time = 24 > 20: TRỄ, bỏ môn dài nhất (10) -> time = 14 | heap = [3, 4, 7]
bước 7: thêm (5, 20) -> time = 19 <= 20: kịp | heap = [3, 4, 5, 7]
đáp án: 4
Bài 1–4 OK


In [28]:
# Bài 5
def chosen_courses(courses):
    order = sorted(range(len(courses)), key=lambda i: courses[i][1])
    h, total = [], 0
    for i in order:
        d, last = courses[i]
        heapq.heappush(h, (-d, i))
        total += d
        if total > last:
            nd, _ = heapq.heappop(h)
            total += nd
    picked = sorted((i for _, i in h), key=lambda i: courses[i][1])
    schedule, t = [], 0
    for i in picked:
        start = t + 1
        t += courses[i][0]
        schedule.append((i, start, t))
    return picked, schedule

picked, sched = chosen_courses(ex1)
print("chọn:", picked)
for i, s, e in sched:
    print(f"  môn {i} {ex1[i]}: học ngày {s} -> {e}")

random.seed(5)
for _ in range(300):
    cs = [[random.randint(1, 8), random.randint(1, 25)] for _ in range(random.randint(1, 9))]
    p, sc = chosen_courses(cs)
    assert len(p) == schedule_brute(cs)
    assert all(e <= cs[i][1] for i, _, e in sc)                       # mọi môn kịp hạn
    assert all(sc[k][1] == (sc[k - 1][2] + 1 if k else 1) for k in range(len(sc)))   # liên tục, không chồng
print("Bài 5 OK")

chọn: [0, 2, 1]
  môn 0 [100, 200]: học ngày 1 -> 100
  môn 2 [1000, 1250]: học ngày 101 -> 1100
  môn 1 [200, 1300]: học ngày 1101 -> 1300
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "lập lịch hạn chót + bỏ việc dài nhất":**

```python
import heapq
jobs.sort(key=lambda j: j.deadline)
heap, total = [], 0
for dur, deadline in jobs:
    total += dur
    heapq.heappush(heap, -dur)
    if total > deadline:
        total += heapq.heappop(heap)     # bỏ việc dài nhất
return len(heap)
```

**Câu thần chú:** *"Xếp theo hạn chót; cứ nhận; trễ thì bỏ môn dài nhất."*

**Checklist trước khi submit:**

- [ ] Sắp theo **`lastDay`** (không phải `duration`).
- [ ] Max-heap: đẩy `-duration`, pop rồi **cộng** vào `total`.
- [ ] Chỉ cần `if` (một lần bỏ) chứ không cần `while`.
- [ ] Đáp án là `len(heap)`.
- [ ] Thử môn quá dài (`d > lastDay`), nhiều môn giống nhau, ví dụ cần thay môn `[[5,5],[4,6],[2,6]]`.
- [ ] Đối chiếu brute force tập con với `n <= 10`.